In [1]:
"""
Median Consensus Clustering (MCC) for graphs
Based on: Hussain et al., "Parallel Algorithms for Median Consensus
Clustering in Complex Networks" (2024).

Reference Python implementation. It prioritizes clarity and fidelity to the
paper's objective over maximum performance. For large graphs, use the authors'
C++/OpenMP implementation (https://github.com/taufique71/pamcon).

Dependencies:
    pip install networkx numpy

Input partitions:
    Each partition may be a dict {node: cluster_label}, or a sequence of
    cluster labels in exactly the same order as `nodes`.
"""

from collections import defaultdict
from itertools import combinations
import networkx as nx
import numpy as np


def _labels_in_node_order(partition, nodes):
    """Convert dict/list partition to a 1-D label array aligned to nodes."""
    if isinstance(partition, dict):
        missing = set(nodes) - set(partition)
        if missing:
            raise ValueError(f"Partition is missing {len(missing)} graph nodes.")
        return np.asarray([partition[v] for v in nodes], dtype=object)

    labels = np.asarray(partition, dtype=object)
    if labels.ndim != 1 or len(labels) != len(nodes):
        raise ValueError("Each sequence partition must have one label per node.")
    return labels


def split_join_distance(labels_a, labels_b, normalized=True):
    """
    Split-join distance between two partitions.

    Raw distance = (n - sum_a max_b overlap(a,b))
                 + (n - sum_b max_a overlap(a,b)).

    The normalized form divides by 2*n and lies in [0, 1].
    """
    a = np.asarray(labels_a, dtype=object)
    b = np.asarray(labels_b, dtype=object)
    if len(a) != len(b):
        raise ValueError("Partitions must contain the same number of vertices.")
    n = len(a)
    if n == 0:
        return 0.0

    counts = defaultdict(int)
    size_a = defaultdict(int)
    size_b = defaultdict(int)
    for x, y in zip(a, b):
        counts[(x, y)] += 1
        size_a[x] += 1
        size_b[y] += 1

    best_a = defaultdict(int)
    best_b = defaultdict(int)
    for (x, y), count in counts.items():
        best_a[x] = max(best_a[x], count)
        best_b[y] = max(best_b[y], count)

    distance = (n - sum(best_a.values())) + (n - sum(best_b.values()))
    return distance / (2.0 * n) if normalized else distance


def homogeneous_partition_groups(partitions, threshold=0.5):
    """
    Group input partitions using the paper's partition-similarity graph:
    partitions are vertices; connect a pair when normalized split-join
    distance <= threshold; return connected components.

    `threshold` is a practical normalized [0,1] parameter. The paper describes
    selecting lambda with an elbow heuristic; it does not prescribe one
    universal value.
    """
    if not partitions:
        raise ValueError("At least one partition is required.")
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("threshold must be between 0 and 1.")

    graph = nx.Graph()
    graph.add_nodes_from(range(len(partitions)))
    for i, j in combinations(range(len(partitions)), 2):
        d = split_join_distance(partitions[i], partitions[j], normalized=True)
        if d <= threshold:
            graph.add_edge(i, j)

    return [sorted(component) for component in nx.connected_components(graph)]


def median_consensus(
    graph,
    partitions,
    max_iterations=100,
    grouping_threshold=None,
    return_groups=False,
):
    """
    Compute graph-aware median consensus.

    Parameters
    ----------
    graph : networkx.Graph
        Undirected graph. Self-loops are ignored.
    partitions : list
        List of dicts {node: label} or label sequences ordered by graph.nodes().
    max_iterations : int
        Safety cap on greedy iterations.
    grouping_threshold : float or None
        If supplied, group homogeneous partitions first and return one
        consensus per connected component. If None, use all partitions
        together, matching the core MCC procedure.
    return_groups : bool
        If True, return (consensus_results, groups).

    Returns
    -------
    dict
        If grouping_threshold is None: {node: consensus_cluster}.
        Otherwise: list of such dictionaries, one per homogeneous group.
    """
    if not isinstance(graph, nx.Graph) or graph.is_directed():
        raise TypeError("graph must be an undirected networkx.Graph.")
    if not partitions:
        raise ValueError("At least one input partition is required.")

    nodes = list(graph.nodes())
    n = len(nodes)
    if n == 0:
        raise ValueError("The graph must contain at least one vertex.")

    # Remove self-loops: they do not create a candidate destination cluster.
    node_to_index = {node: i for i, node in enumerate(nodes)}
    adjacency = [
        [node_to_index[w] for w in graph.neighbors(v)
         if node_to_index[w] != i]
        for i, v in enumerate(nodes)
    ]

    aligned = [_labels_in_node_order(p, nodes) for p in partitions]

    if grouping_threshold is None:
        groups = [list(range(len(aligned)))]
    else:
        groups = homogeneous_partition_groups(aligned, grouping_threshold)

    results = []
    for group in groups:
        group_partitions = [aligned[i] for i in group]
        result = _single_group_consensus(
            nodes, adjacency, group_partitions, max_iterations
        )
        results.append(result)

    output = results[0] if grouping_threshold is None else results
    if return_groups:
        return output, groups
    return output


def _single_group_consensus(nodes, adjacency, partitions, max_iterations):
    n = len(nodes)
    k = len(partitions)

    # C[u, p] is the input cluster label of vertex u in partition p.
    membership = np.column_stack(partitions)

    # Start from singleton clusters, as specified in Section 2.4.
    consensus = np.arange(n, dtype=np.int64)

    for _iteration in range(max_iterations):
        # Snapshot cluster membership for this iteration. All proposed moves
        # are evaluated against the same snapshot (paper's parallel strategy).
        members = defaultdict(list)
        for u, cluster in enumerate(consensus):
            members[int(cluster)].append(u)

        proposed = {}

        for v in range(n):
            current = int(consensus[v])

            # Only clusters containing a graph neighbor are candidate targets.
            candidate_clusters = {
                int(consensus[u]) for u in adjacency[v]
                if int(consensus[u]) != current
            }
            if not candidate_clusters:
                continue

            # delta_uv = number of input partitions separating u and v.
            # Weight w_uv = 2*delta_uv - k, as in Eq. (5).
            def cluster_score(cluster_id, exclude_vertex=None):
                total = 0
                for u in members.get(cluster_id, ()):
                    # Pair (v, v) is not part of the partition objective.
                    if u == exclude_vertex:
                        continue
                    delta_uv = int(np.count_nonzero(membership[u] != membership[v]))
                    total += 2 * delta_uv - k
                return total

            current_score = cluster_score(current, exclude_vertex=v)
            best_delta = 0
            best_cluster = None

            for target in sorted(candidate_clusters):
                delta_objective = cluster_score(target) - current_score
                if delta_objective < best_delta:
                    best_delta = delta_objective
                    best_cluster = target

            if best_cluster is not None:
                proposed[v] = best_cluster

        if not proposed:
            break

        # Apply all independently computed moves together, matching the
        # synchronous update strategy described in Section 2.5.
        for v, target in proposed.items():
            consensus[v] = target

    # Relabel cluster IDs compactly as 0, 1, 2, ...
    unique = {}
    compact = []
    for cluster in consensus:
        cluster = int(cluster)
        if cluster not in unique:
            unique[cluster] = len(unique)
        compact.append(unique[cluster])

    return {node: compact[i] for i, node in enumerate(nodes)}


if __name__ == "__main__":
    # Small runnable example
    G = nx.karate_club_graph()

    # Three example input partitions. In a real experiment, replace these
    # with outputs from Louvain/Leiden/other community detection algorithms.
    nodes = list(G.nodes())
    p1 = {v: (0 if v < 17 else 1) for v in nodes}
    p2 = {v: (0 if v < 16 else 1) for v in nodes}
    p3 = {v: (0 if v in G.nodes() and v % 2 == 0 else 1) for v in nodes}

    consensus = median_consensus(G, [p1, p2, p3])
    number_of_clusters = len(set(consensus.values()))
    print("Number of consensus clusters:", number_of_clusters)
    print("Consensus assignment:", consensus)

    # Optional: first group similar input partitions, then compute one
    # independent consensus for each homogeneous group.
    grouped_results, groups = median_consensus(
        G,
        [p1, p2, p3],
        grouping_threshold=0.5,
        return_groups=True,
    )
    print("Homogeneous groups (partition indices):", groups)
    print("Number of group consensuses:", len(grouped_results))


Number of consensus clusters: 6
Consensus assignment: {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0, 13: 0, 14: 1, 15: 2, 16: 0, 17: 3, 18: 4, 19: 4, 20: 4, 21: 5, 22: 4, 23: 4, 24: 4, 25: 4, 26: 4, 27: 4, 28: 4, 29: 4, 30: 4, 31: 4, 32: 4, 33: 4}
Homogeneous groups (partition indices): [[0, 1, 2]]
Number of group consensuses: 1
